# Stage 0: System Prompts Chat

A chat interface for experimenting with system prompts.

**Features:**
- Uses Stage 1 Baseline RAG agent as foundation
- Allows dynamic system prompt override via `set_system_instructions()`
- Experiment with different personas and instructions

## Setup

Run this cell once to initialize the agent.

In [ ]:
import sys
import os
from pathlib import Path

# =============================================================================
# ENVIRONMENT-BASED CONFIGURATION (Auto-detection)
# Set LOCAL_DEV=true in your environment for local development
# =============================================================================
LOCAL_DEV = os.environ.get("LOCAL_DEV", "false").lower() in ("true", "1", "yes")

if LOCAL_DEV:
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379")
    os.environ.setdefault("OPENAI_API_BASE", "http://localhost:4000")
    print("Running in LOCAL_DEV mode (using localhost)")
else:
    os.environ.setdefault("REDIS_URL", "redis://redis:6379")
    print("Running in PS Portal mode (using Docker network)")

# Map OPENAI_API_BASE to OPENAI_BASE_URL for LiteLLM compatibility
if "OPENAI_API_BASE" in os.environ:
    os.environ["OPENAI_BASE_URL"] = os.environ["OPENAI_API_BASE"]

# Enable nested event loops for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Setup paths (from chat_notebooks, go up two levels to ws/)
project_root = Path("../..").resolve()
stage1_path = project_root / "progressive_agents" / "stage1_baseline_rag"
src_path = project_root / "src"
sys.path.insert(0, str(stage1_path))
sys.path.insert(0, str(src_path))

# Initialize agent
from agent import setup_agent
from agent.nodes import set_system_instructions
from agent.chat_interface import start_chat

print("Initializing Stage 0 System Prompts Agent...")
workflow, course_manager = setup_agent(auto_load_courses=True, verbose=False)

print("Agent ready!")

## Set System Instructions

Modify the system prompt below and run the cell to change the agent's behavior.

In [ ]:
# Example: Minimal system prompt
system_prompt = "You are a helpful assistant."

# Override the agent's system instructions
set_system_instructions(system_prompt)
print(f"System instructions set to:\n{system_prompt}")

## Start Chat

Run this cell to start chatting with the agent using the current system instructions.

In [ ]:
start_chat(workflow)

---

## Example System Prompts to Try

Copy any of these into the cell above and re-run to experiment:

### Minimal
```python
system_prompt = "You are a helpful assistant."
```

### Role-Focused
```python
system_prompt = """You are a Redis University academic advisor.
Help students find courses that match their interests and goals."""
```

### Structured (Recommended)
```python
system_prompt = """<role>
You are a Redis University academic advisor helping students navigate course offerings.
</role>

<capabilities>
- Search and recommend courses based on student interests
- Explain course prerequisites and requirements
- Suggest learning paths for different career goals
</capabilities>

<constraints>
- Only recommend courses from the Redis University catalog
- Be concise but thorough in explanations
- If unsure, acknowledge limitations
</constraints>"""
```